In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(torch.cuda.device_count())


x = torch.randn(3, 3, device="cuda")
y = x @ x
print(x)
print(y)



1
tensor([[ 0.8790,  0.0135,  0.0550],
        [ 0.6164,  1.0914, -1.0214],
        [-0.1869, -2.7670,  0.3756]], device='cuda:0')
tensor([[ 0.7706, -0.1257,  0.0553],
        [ 1.4054,  4.0258, -1.4645],
        [-1.9401, -4.0617,  2.9571]], device='cuda:0')


/home/htamm/miniconda3/envs/agront/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

# Get pretrained model
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name="1B_agro_nt",
    embeddings_layers_to_save=(12,),
    max_positions=32,
)
forward_fn = hk.transform(forward_fn)


/home/htamm/miniconda3/envs/agront/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded model's hyperparameters.
Downloaded model's weights...


In [2]:
# Get data and tokenize it
sequences = ["NGGACAGCGG", "NGGACGGCGG"]
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

# Initialize random key
random_key = jax.random.PRNGKey(0)

# Inference
outs = forward_fn.apply(parameters, random_key, tokens)

# Get embeddings at layer 20
print(outs["embeddings_12"].shape)

(2, 32, 1500)


In [3]:
print(outs["embeddings_12"])

[[[ -0.37829465  -7.817052     1.2317715  ...   2.403119     0.26740313
    -5.193776  ]
  [  3.6248162  -11.280587     9.808967   ...  -0.63659656  -5.446619
    -2.5782893 ]
  [  2.073442    -7.654725     6.6665025  ...   1.7031863    2.5718224
    -2.1831853 ]
  ...
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]]

 [[ -0.9202825   -6.171276     2.5364666  ...   3.795408     2.1836684
    -3.2639008 ]
  [  4.171793   -10.664946    12.369383   ...   1.0581734   -2.5833874
    -2.0383883 ]
  [  0.91910446  -5.6572685   12.832234   ...   4.4644156    3.8622031
    -2.9300437 ]
  ...
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5